In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error
import re
import csv

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        holdout = candidate / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_DLLM' / 'heloc_DLLM_holdout.csv'
        if holdout.exists():
            return candidate
    raise FileNotFoundError('Impossibile trovare la radice del progetto.')

project_root = find_project_root()
clean_test_path = project_root / 'data' / 'processed' / 'Fase2' / 'SplitDataset' / 'Split_DLLM' / 'heloc_DLLM_imputation_test.csv'
mask_dir = project_root / 'data' / 'processed' / 'Fase2' / 'DataCorruption' / 'heloc_DLLM'
imputated_root = project_root / 'data' / 'processed' / 'Fase3' / 'Imputated_DLLM'
output_dir = project_root / 'data' / 'processed' / 'Fase3' / 'Results'

print(f'Project root : {project_root}')
print(f'Clean Test   : {clean_test_path.exists()}')
print(f'Mask Dir     : {mask_dir.exists()}')
print(f'Imputated Dir: {imputated_root.exists()}')
print(f'Output Dir   : {output_dir}')

Project root : /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project
Clean Test   : True
Mask Dir     : True
Imputated Dir: True
Output Dir   : /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Results


In [2]:
# 1. Carichiamo il dataset pulito (Ground Truth)
df_clean = pd.read_csv(clean_test_path)

# La variabile target (RiskPerformance) non viene corrotta né imputata, quindi la ignoriamo.
if 'RiskPerformance' in df_clean.columns:
    df_clean = df_clean.drop(columns=['RiskPerformance'])

# Identifichiamo le colonne numeriche nel dataset DLLM.
# Le colonne categoriche (stringhe semantiche come 'Never Had Delinquency') vengono escluse
# perché MSE/MAE non hanno significato su valori categorici.
numeric_cols = []
for col in df_clean.columns:
    converted = pd.to_numeric(df_clean[col], errors='coerce')
    if converted.notna().mean() > 0.8:
        numeric_cols.append(col)

print(f'Colonne totali (senza target): {len(df_clean.columns)}')
print(f'Colonne numeriche selezionate: {len(numeric_cols)}')
print(f'Colonne categoriche escluse  : {len(df_clean.columns) - len(numeric_cols)}')
if len(df_clean.columns) - len(numeric_cols) > 0:
    cat_cols = [c for c in df_clean.columns if c not in numeric_cols]
    print(f'  \u2192 {cat_cols}')

# Convertiamo le colonne numeriche a float nel ground truth
df_clean_num = df_clean[numeric_cols].apply(pd.to_numeric, errors='coerce')

results = []

# Per la pipeline DLLM i file imputati sono direttamente in Imputated_DLLM/ (senza sottocartelle per metodo)
imputed_files = sorted(imputated_root.glob('*_discriminative_train_*.csv'))

for imp_path in imputed_files:
    # Estrarre strategia e percentuale dal nome file
    m = re.match(r'^(.+?)_discriminative_train_([A-Z]+)_(\d+)\.csv$', imp_path.name)
    if not m:
        continue
    dataset, strategy, pct = m.groups()
    
    # 2. Caricare il dataset imputato
    df_imputed = pd.read_csv(imp_path)
    if 'RiskPerformance' in df_imputed.columns:
        df_imputed = df_imputed.drop(columns=['RiskPerformance'])
        
    # 3. Caricare la corrispondente maschera di valori mancanti (True = valore mancante)
    mask_name = f'{dataset}_imputation_test_mask_{strategy}_{pct}.csv'
    mask_path = mask_dir / mask_name
    df_mask = pd.read_csv(mask_path)
    if 'RiskPerformance' in df_mask.columns:
        df_mask = df_mask.drop(columns=['RiskPerformance'])
        
    # Assicuriamoci che i dataset abbiano la stessa forma
    assert df_clean.shape == df_imputed.shape == df_mask.shape
    
    # 4. Restringiamo alle sole colonne numeriche
    clean_vals = df_clean_num.values
    imputed_vals = df_imputed[numeric_cols].apply(pd.to_numeric, errors='coerce').values
    mask_vals = df_mask[numeric_cols].values
    
    # 5. Calcoliamo MSE/MAE esclusivamente sui valori mancanti nelle colonne numeriche
    clean_missing_only = clean_vals[mask_vals]
    imputed_missing_only = imputed_vals[mask_vals]
    
    # Rimuoviamo eventuali NaN residui (conversioni fallite)
    valid_mask = ~(np.isnan(clean_missing_only) | np.isnan(imputed_missing_only))
    clean_valid = clean_missing_only[valid_mask]
    imputed_valid = imputed_missing_only[valid_mask]
    
    mse = mean_squared_error(clean_valid, imputed_valid)
    mae = mean_absolute_error(clean_valid, imputed_valid)
    
    results.append({
        'imputation_method': 'DLLM',
        'dataset': dataset,
        'missing_strategy': strategy,
        'missing_pct': int(pct),
        'mse': mse,
        'mae': mae
    })
    print(f'DLLM | {strategy} {pct}% -> MSE: {mse:.4f}, MAE: {mae:.4f}')

Colonne totali (senza target): 23
Colonne numeriche selezionate: 20
Colonne categoriche escluse  : 3
  → ['MSinceMostRecentDelq', 'MSinceMostRecentInqexcl7days', 'NetFractionInstallBurden']
DLLM | MAR 10% -> MSE: 2153.6469, MAE: 17.1308
DLLM | MAR 25% -> MSE: 1639.5074, MAE: 17.4595
DLLM | MAR 40% -> MSE: 2436.0636, MAE: 23.8260
DLLM | MCAR 10% -> MSE: 2356.6081, MAE: 17.6193
DLLM | MCAR 25% -> MSE: 349.2195, MAE: 8.9494
DLLM | MCAR 40% -> MSE: 4335.5842, MAE: 19.2766
DLLM | MNAR 10% -> MSE: 1519.7717, MAE: 17.5306
DLLM | MNAR 25% -> MSE: 4787.7260, MAE: 21.8180
DLLM | MNAR 40% -> MSE: 4881.2540, MAE: 21.9536


In [3]:
# 6. Salviamo i risultati
output_dir.mkdir(parents=True, exist_ok=True)
report_path = output_dir / 'baseline_imputation_mse_mae_DLLM.csv'

df_results = pd.DataFrame(results)
df_results.to_csv(report_path, index=False)

print(f'\nRisultati di Baseline salvati in: {report_path}')

# Mostriamo la tabella finale
df_results.sort_values(by=['missing_strategy', 'missing_pct'])


Risultati di Baseline salvati in: /Users/marco/Documents/Biometry/DLL/DLLM_Project/DLLM_Project/data/processed/Fase3/Results/baseline_imputation_mse_mae_DLLM.csv


,imputation_method,dataset,missing_strategy,missing_pct,mse,mae
0,DLLM,heloc_DLLM,MAR,10,2153.646930,17.130815
1,DLLM,heloc_DLLM,MAR,25,1639.507443,17.459521
2,DLLM,heloc_DLLM,MAR,40,2436.063636,23.826049
3,DLLM,heloc_DLLM,MCAR,10,2356.608077,17.619262
4,DLLM,heloc_DLLM,MCAR,25,349.219474,8.949370
5,DLLM,heloc_DLLM,MCAR,40,4335.584201,19.276627
6,DLLM,heloc_DLLM,MNAR,10,1519.771725,17.530558
7,DLLM,heloc_DLLM,MNAR,25,4787.725993,21.818010
8,DLLM,heloc_DLLM,MNAR,40,4881.254049,21.953608
